In [1]:
import os
import dask.dataframe as dd
import pandas as pd
from utilities import save_df_in_chunks

import warnings
warnings.filterwarnings('ignore')

from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=6)
client = Client(cluster)  # Connect to distributed cluster and override default
client

Client Scheduler: tcp://127.0.0.1:52542 Dashboard: http://127.0.0.1:52541/status,Cluster Workers: 6 Cores: 12 Memory: 34.29 GB


In [2]:
RECORDS_PATH =  os.path.join('..', 'raw-files', 'sg_subzones_devices')
SAVEPATH = os.path.join('intermediate-files', 'deviceID-list')
DEVICE_COUNT_PATH = os.path.join('intermediate-files', 'tt_appearances_areadate', '*.csv.gz')

In [3]:
# Get device ID dict
devices_ddf = (dd.read_csv(DEVICE_COUNT_PATH, compression='gzip', usecols=['deviceID'])
              ).compute()

devices_ddf.sort_values('deviceID', inplace=True)
devices_ddf.reset_index(drop=True, inplace=True)
devices = dict(zip(devices_ddf['deviceID'], devices_ddf.index))
# https://stackoverflow.com/questions/19201290/how-to-save-a-dictionary-to-a-file/32216025
# np.save(os.path.join('intermediate-files', 'device-xid.npy'), devices)

In [4]:
devices = (pd.DataFrame.from_dict(devices, orient='index').reset_index()
           .rename(columns={'index': 'hashid', 0: 'xid'})
           .astype({'xid': 'int'}))

In [5]:
devices

,hashid,xid
0,M200M21yaHQybmd0OTpoNzQ5dTFwYjMzN2g=,0
1,M200M21yaHQybmd0cDpoNzQ5dHVwYjMzN3Q=,1
2,M200M21yaHQybmd0cDpoNzQ5dHZwYjMzODg=,2
3,M200M21yaHQybmd1OTpoNzQ5dTBwYjMzOGo=,3
4,M200M21yaHQybmd1cDpoNzQ5dTFwYjMzOHU=,4
...,...,...
17492184,dnZzaW1tdmFubGtjOjZxdGNoMjVuNXRnbXI=,17492184
17492185,dnZzaW1tdmFubGtzOjZxdGNoMjZuNXRnbjY=,17492185
17492186,dnZzaW1tdmFubGxjOjZxdGNoMjNuNXRnbmk=,17492186
17492187,dnZzbjBmYzBpajl2OmNsYTVzdWV0cjFwNDk=,17492187


In [6]:
save_df_in_chunks(devices, SAVEPATH, 5, 'deviceID-list-chunk', 'csv.gz', compression='gzip')